In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import (HumanMessage, AIMessage, ToolMessage )
from langchain_core.tools import tool
from langgraph.graph import ( StateGraph, START, END)
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq

In [ ]:
import os
from dotenv import load_dotenv

# Load API key from backend/.env
dotenv_path = os.path.abspath("../../../../.env")
load_dotenv(dotenv_path)

if not os.environ.get("GROQ_API_KEY"):
    print("Warning: GROQ_API_KEY not found in backend/.env")
else:
    print("GROQ_API_KEY loaded successfully.")

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [ ]:
#  tool binding to the llm
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# Prompt
SYSTEM_PROMPT = """
You are the Merchant Commerce Agent.

You are responsible for handling customer shopping requests.

You have access to commerce tools.

Rules:

1. Never claim that a product was added unless add_to_cart
   successfully confirms it.

2. Never directly modify the cart yourself.

3. Use get_cart when you need current cart information.

4. Use get_upsell_products when the customer asks for
   recommendations or when an appropriate complementary
   product can be suggested.

5. Do not invent products.

6. Keep responses concise and commerce-focused.

7. If a tool fails, clearly tell the customer that the
   requested operation could not be completed.

8. MUST: Do not add the item to the cart until and unless customer ask to do, if he is asking about the product just tell it  where you have or not,
    and then ask him if he would like it to be added to the cart and along with it recommend the items that this inventory have
"""

In [ ]:
class CommerceState(TypedDict):

    messages: Annotated[ list, add_messages ]

""" later add ons
user_id
merchant_id
cart
payment_context
policy_context
"""

In [ ]:
def merchant_llm_node(state: CommerceState):

    messages = state["messages"]

    system_message = {
        "role": "system",
        "content": SYSTEM_PROMPT
    }

    response = llm_with_tools.invoke(
        [system_message] + messages
    )

    return {
        "messages": [response]
    }

    

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(CommerceState)

# nodes
graph.add_node("merchant_llm", merchant_llm_node )
graph.add_node( "tools", tool_node )

# edges
graph.add_edge(START, "merchant_llm" )
graph.add_conditional_edges("merchant_llm", tools_condition)
graph.add_edge("tools", "merchant_llm")


workflow = graph.compile()